# MEOK · Per-Hive Embodied Policies — MuJoCo Playground (MJX) on free GPU

Train one RL **control policy per hive** on a **free Colab/Kaggle GPU** using
[MuJoCo Playground](https://github.com/google-deepmind/mujoco_playground) (MJX + Brax PPO).

**Honest scope (read this):**
- This trains *embodied/locomotion* policies (the robotics/embodied-gov arm) — it is **NOT** the
  society-sim that produces the flywheel's governed-vs-ungoverned `A_crimes`/`B_crimes`.
  Those come from `sovereign-town`. This adds a **separate** signed record type: per-hive policy metrics.
- Runtime > 0 cost is GPU time only (free tier). On a Colab **T4**, a small env trains in minutes-to-~1h.
- The exact Playground API moves fast — if a call errors, check the repo's `learning/` notebooks for the
  installed version. The per-hive harness + export below is the durable part.
- Output `meok_hive_policies.json` is **ready for the King to Ed25519-sign** into the ledger
  (same scheme as the flywheel: `prev` + canonical json). Signing happens on the King host, not here.


## 1. Runtime check + install


In [ ]:
# Runtime > Change runtime type > GPU (T4 is fine).
!nvidia-smi -L || echo 'No GPU — set Runtime type to GPU'


In [ ]:
# Colab usually ships JAX+CUDA. Install MuJoCo Playground (+ deps).
!pip install -q mujoco_playground brax
# If JAX can't see the GPU on Kaggle, uncomment:
# !pip install -q --upgrade "jax[cuda12]"


In [ ]:
import jax, time, json, functools
print('JAX devices:', jax.devices())
assert any(d.platform=='gpu' for d in jax.devices()), 'No GPU visible — switch runtime to GPU'


## 2. The 12 hives → per-hive training config
Each hive gets its own seed + env + a **governed** flag (safety-constrained reward).
`governed=True` hives get an action-rate / fall penalty (the 'sovereign' constraint);
`governed=False` is the unconstrained baseline — that contrast is the story we surface on MEOK Earth.


In [ ]:
HIVES = [
  {'id':'meok',                'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':0},
  {'id':'proofof',             'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':1},
  {'id':'councilof',           'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':2},
  {'id':'safetyof',            'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':3},
  {'id':'openmoe',             'env':'Go1JoystickFlatTerrain', 'governed':False, 'seed':4},
  {'id':'transparencyof',      'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':5},
  {'id':'accountabilityof',    'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':6},
  {'id':'dataprivacyof',       'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':7},
  {'id':'ethicalgovernanceof', 'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':8},
  {'id':'biasdetectionof',     'env':'Go1JoystickFlatTerrain', 'governed':True,  'seed':9},
  {'id':'agisafe',             'env':'Go1JoystickFlatTerrain', 'governed':False, 'seed':10},
  {'id':'asisecurity',         'env':'Go1JoystickFlatTerrain', 'governed':False, 'seed':11},
]
# Start with 1-2 hives to confirm the loop, then scale. Heavy: 12 x full runs.
TRAIN_HIVES = HIVES[:2]
# Demo budget — raise num_timesteps for real policies (e.g. 60_000_000).
TIMESTEPS = 3_000_000


## 3. Train one hive (MJX + Brax PPO)


In [ ]:
from mujoco_playground import registry
from mujoco_playground.config import locomotion_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

def train_hive(hive, timesteps=TIMESTEPS):
    env_name = hive['env']
    env       = registry.load(env_name)
    eval_env  = registry.load(env_name)
    cfg = locomotion_params.brax_ppo_config(env_name)
    # override budget + seed for this hive
    cfg.num_timesteps = timesteps
    cfg.seed          = hive['seed']
    # 'governed' hives keep the env's safety/energy penalties on (default);
    # baseline hives relax them so the contrast is visible.
    if not hive['governed'] and hasattr(env, 'unsafe_mode'):
        env.unsafe_mode = True
    progress = {}
    def cb(step, metrics):
        progress[int(step)] = float(metrics.get('eval/episode_reward', float('nan')))
        print(f"  {hive['id']} step={step:>9} reward={progress[int(step)]:.2f}")
    make_networks = functools.partial(ppo_networks.make_ppo_networks,
                                       **cfg.network_factory) if 'network_factory' in cfg else ppo_networks.make_ppo_networks
    train_fn = functools.partial(ppo.train,
        **{k:v for k,v in dict(cfg).items() if k!='network_factory'},
        network_factory=make_networks, progress_fn=cb)
    t0=time.time()
    _, _, metrics = train_fn(environment=env, eval_env=eval_env)
    return {'reward': float(metrics.get('eval/episode_reward', float('nan'))),
            'wall_s': round(time.time()-t0,1), 'curve': progress}


## 4. Train the selected hives + collect metrics


In [ ]:
results=[]
for h in TRAIN_HIVES:
    print(f"=== training hive {h['id']} (governed={h['governed']}) ===")
    try:
        m = train_hive(h)
    except Exception as e:
        print('  ! train error (check Playground API for this version):', e)
        m = {'reward': None, 'wall_s': None, 'error': str(e)}
    results.append({**h, **m})
results


## 5. Export signed-ready metrics → `meok_hive_policies.json`
This file goes to the King host; the King Ed25519-signs it into the ledger as a
`hive_policy` record (same canonical scheme as the flywheel). MEOK Earth then reads
it as a new layer (per-hive policy reward + governed flag).


In [ ]:
import datetime
export = {
  'kind': 'hive_policy_batch',
  'engine': 'mujoco_playground/mjx + brax_ppo',
  'exported_at': datetime.datetime.utcnow().isoformat()+'Z',
  'timesteps_per_hive': TIMESTEPS,
  'hives': [{'id':r['id'],'env':r['env'],'governed':r['governed'],
             'reward':r.get('reward'),'wall_s':r.get('wall_s')} for r in results],
}
with open('meok_hive_policies.json','w') as f: json.dump(export,f,indent=2)
print(json.dumps(export, indent=2))
from google.colab import files  # on Kaggle: use the Output panel instead
files.download('meok_hive_policies.json')


## 6. Feed into MEOK Earth (on the King host, not here)
```bash
# 1) copy meok_hive_policies.json to the King host
# 2) King signs it into the ledger (prev + canonical json, Ed25519) via sign_lib.py
# 3) export to the globe:  npm run export:ledger  (extend it to merge hive_policy rows)
# 4) MEOK Earth shows a 'Policies' layer: per-hive reward, governed vs baseline
```
Each hive now carries a **trained, attestable policy** — 'more neural nets per hive', on free GPU,
with the governed-vs-baseline contrast preserved end-to-end into the signed ledger.
